In [2]:
import os, json, warnings
import numpy as np
import xarray as xr
import proplot as pplt
warnings.filterwarnings('ignore')

pplt.rc.update({
    'savefig.dpi':900, 'savefig.bbox':'tight', 'savefig.pad_inches':0.02,
    'tick.minor':False, 'font.size':9, 'label.size':9, 'tick.labelsize':9,
    'title.size':9, 'abc.size':9, 'legend.fontsize':9, 'suptitle.size':9})

with open('../scripts/configs.json') as f:
    CONFIGS = json.load(f)
SPLITSDIR  = CONFIGS['filepaths']['splits']
WEIGHTSDIR = CONFIGS['filepaths']['weights']
PREDSDIR   = CONFIGS['filepaths']['predictions']
SEEDS      = CONFIGS['experiments']['nn']['seeds']
SPLIT      = 'test'

with open(os.path.join(SPLITSDIR, 'stats.json')) as f:
    STATS = json.load(f)
TP_MEAN = STATS['tp_mean']
TP_STD  = STATS['tp_std']
ZMIN    = (0.0 - TP_MEAN) / TP_STD

In [3]:
# Load test split: kernel-integrate field vars, load surface vars
splitds   = xr.open_dataset(os.path.join(SPLITSDIR, f'norm_{SPLIT}.h5'), engine='h5netcdf')
refda     = splitds['tp'].transpose('time', 'lat', 'lon')
ntime, nlat, nlon = splitds.sizes['time'], splitds.sizes.get('lat',1), splitds.sizes.get('lon',1)

fieldvars = ['rh', 'thetae', 'thetaestar']
localvars = ['lf', 'shf', 'lhf', 'sef', 'sdo', 'se']
nsig      = splitds.sizes['sig']
dsig      = splitds['dsig'].values
fields    = np.stack([splitds[v].transpose('time','lat','lon','sig').values.reshape(-1, nsig)
                      for v in fieldvars], axis=1)
surfmask  = (splitds['surfmask'].transpose('time','lat','lon','sig').values.reshape(-1, nsig)
             if 'surfmask' in splitds else None)

def kernel_integrate(fields, weights, dsig, mask=None):
    w = fields * weights[None,:,:] * dsig[None,None,:]
    if mask is not None: w *= mask[:,None,:]
    return w.sum(axis=2)

kint = np.mean([kernel_integrate(
                    fields,
                    xr.open_dataset(os.path.join(WEIGHTSDIR, f'nn_gauss_{s}_weights.nc'),
                                    engine='h5netcdf')['k'].values,
                    dsig, surfmask)
                for s in SEEDS], axis=0)   # (nsamples, 3)

FEATS = {v: kint[:, i] for i, v in enumerate(fieldvars)}
for v in localvars:
    da = splitds[v]
    FEATS[v] = (da.transpose('time','lat','lon').values.ravel() if 'time' in da.dims
                else np.tile(da.values, (ntime,1,1)).ravel())
splitds.close()

# Native-unit true precipitation
with xr.open_dataset(os.path.join(SPLITSDIR, f'{SPLIT}.h5'), engine='h5netcdf') as ds:
    TRUETP = ds.tp.load()

print('Features loaded.')

Features loaded.


In [4]:
SRFN = dict(cube=lambda x:x**3, square=lambda x:x**2, neg=lambda x:-x,
            exp=np.exp, log=np.log, abs=np.abs, sqrt=np.sqrt,
            max=np.maximum, min=np.minimum)

def eval_sr(form, extra={}):
    ns = dict(SRFN, __builtins__={}, **FEATS, **extra)
    return np.asarray(eval(form, ns), dtype=float)

def pred_to_da(arr_mm):
    """Reshape flat mm array → DataArray with refda coordinates."""
    return xr.DataArray(arr_mm.reshape(ntime, nlat, nlon),
                        dims=refda.dims, coords=refda.coords)

def sr_to_mm(expr_out):
    """Convert PySR excess-above-zmin output → native mm."""
    z = ZMIN + np.maximum(expr_out, 0)
    return np.maximum(np.expm1(z * TP_STD + TP_MEAN), 0)

def get_r2(pred_mm_da):
    ytrue, ypred = xr.align(TRUETP, pred_mm_da, join='inner')
    ss_res = ((ytrue - ypred)**2).sum(skipna=True)
    ss_tot = ((ytrue - ytrue.mean(skipna=True))**2).sum(skipna=True)
    return float(1 - ss_res / ss_tot)

# SR-MED baseline
srmed_raw = eval_sr('a * cube(max(rh, thetae - b * thetaestar - c))',
                    {'a':1.5576, 'b':1.4706, 'c':0.3756})

# NN-GAUSS predictions (already native mm)
with xr.open_dataset(os.path.join(PREDSDIR, f'nn_gauss_{SPLIT}_predictions.nc'), engine='h5netcdf') as ds:
    nn_gauss_mm = ds.tp.mean('seed').load()

print('Baselines ready.')

Baselines ready.


In [ ]:
# Candidates from the current sr_gauss_resid run (maxsize=30). PySR sees only
# {srmed, lf, shf, lhf, sef, sdo, se}, so every equation below is an srmed correction.
#
# Cross-seed structure at c19-30 (see models/sr/sr_gauss_resid_{42,72,102}_equations.csv):
#   motif 1  lf-gated shf product, all 3 seeds
#              s42  shf * min(lf, b) * a
#              s72  (shf - c) * cube(lf - b)
#              s102 max(neg(srmed), shf) * (lf - b)
#   motif 2  negative sef term, all 3 seeds; at c25-30 seeds 42 and 102 both wrap it
#            as max(., cube(sef)), while s72 scales it by cube(lf)
#   motif 3  linear negative lhf, all 3 seeds at c6, resurfacing at s42 c22 / s102 c17-19
#
# The `cons-*` entries are hand-written consensus forms with constants copied verbatim
# from one supporting seed (NOT refit) — their R2 is a lower bound until optimize.py
# runs L-BFGS-B multistart on them.

RESID = {'srmed': srmed_raw}

candidates = [
    # --- baselines ---
    dict(label='SR-MED',     color='#D42028', pred_mm=pred_to_da(sr_to_mm(srmed_raw))),
    dict(label='SR-HI',      color='#8B0000',
         pred_mm=pred_to_da(sr_to_mm(eval_sr(
             'cube(a * max(rh, thetae + b * thetaestar + c) + max(lf, shf) * d)',
             {'a':1.307, 'b':-1.3594, 'c':-0.4173, 'd':-0.1271})))),
    dict(label='NN-GAUSS',   color='#2355a1', pred_mm=nn_gauss_mm),

    # --- superseded designs (train.py code for these runs has been removed) ---
    dict(label='resid c19 (v1)',  color='#f4a261',
         pred_mm=pred_to_da(sr_to_mm(eval_sr(
             'srmed + min((exp(shf) + -1.1003046) * (lf + -1.2116383), 1.1753592 - lhf)',
             RESID)))),
    dict(label='gap c20 (v1)',    color='#e76f51',
         pred_mm=pred_to_da(sr_to_mm(eval_sr(
             '(srmed + (min(lf, 0.106686234) * (shf + (shf * srmed)))) - (lhf * 0.18459359)',
             RESID)))),
    dict(label='distil c20 (v1)', color='#457b9d',
         pred_mm=pred_to_da(sr_to_mm(eval_sr(
             'cube(max(rh, (thetaestar * -1.4840289) + (thetae + -0.2807161)) - (rh * -0.1370878))')))),

    # --- verbatim PySR equations, current run ---
    # s42 c20: saturating lf gate on shf, sef clipped from below
    dict(label='s42 c20',    color='#ffb703',
         pred_mm=pred_to_da(sr_to_mm(eval_sr(
             '(srmed + (shf * (min(0.09094131, lf) * 2.9731739))) - max(-0.16219175, sef + -0.9121144)',
             RESID)))),
    # s42 c30: lowest loss of seed 42; cube(sef) clipped against a sdo branch
    dict(label='s42 c30',    color='#fb8500',
         pred_mm=pred_to_da(sr_to_mm(eval_sr(
             '(((shf * min(lf, 0.16941603)) * -2.778827) - (srmed - ((max(cube(sef), neg(sdo + 0.5723901)) + -0.6879842) * 0.26971793))) * -0.9826296',
             RESID)))),
    # s72 c23: cubic lf gate, sef scaled by cube(lf)
    dict(label='s72 c23',    color='#90be6d',
         pred_mm=pred_to_da(sr_to_mm(eval_sr(
             '(srmed - ((0.09104345 - shf) * cube(lf - 0.62423956))) - ((cube(lf) * sef) * 0.12263883)',
             RESID)))),
    # s72 c30: lowest loss of seed 72; same core wrapped in a max floor
    dict(label='s72 c30',    color='#43aa8b',
         pred_mm=pred_to_da(sr_to_mm(eval_sr(
             'max((srmed * 0.40198317) - -0.076174915, srmed - (((0.09825759 - shf) - ((cube(lf) * sef) * -0.1341422)) * cube(lf - 0.5725921)))',
             RESID)))),
    # s102 c21: shf floored at -srmed, sef clipped from below
    dict(label='s102 c21',   color='#f9c74f',
         pred_mm=pred_to_da(sr_to_mm(eval_sr(
             'srmed + (((max(neg(srmed), shf) * (lf - 1.0400712)) + 0.879664) - max(0.69717574, sef))',
             RESID)))),
    # s102 c25: lowest loss of seed 102; max(., cube(sef)) shared with s42 c26-30
    dict(label='s102 c25',   color='#f8961e',
         pred_mm=pred_to_da(sr_to_mm(eval_sr(
             '(srmed + (max(neg(srmed), shf) * (lf + -0.9266265))) + ((max(-0.43502578, cube(sef)) - 0.60446024) * -0.30570245)',
             RESID)))),

    # --- consensus forms (constants copied from a supporting seed, not refit) ---
    # cons-lin c20: seeds 42 (c19) + 102 (c15/c19); linear sef, saturating lf gate
    dict(label='cons-lin c20',  color='#577590',
         pred_mm=pred_to_da(sr_to_mm(eval_sr(
             'srmed + a * min(lf, b) * (shf - c) - d * sef',
             dict(RESID, a=2.3545923, b=0.1330351, c=0.062100567, d=0.18784371))))),
    # cons-cube c26: seeds 42 (c26-c30) + 102 (c25/c27); clipped cube(sef)
    dict(label='cons-cube c26', color='#277da1',
         pred_mm=pred_to_da(sr_to_mm(eval_sr(
             'srmed + a * shf * min(lf, b) - c * (max(d, cube(sef)) - e)',
             dict(RESID, a=2.6748257, b=0.19116789, c=0.28368908, d=-0.43502578, e=0.69079))))),
    # cons-full c30: cons-cube plus the motif-3 lhf term present in all three seeds
    dict(label='cons-full c30', color='#4d908e',
         pred_mm=pred_to_da(sr_to_mm(eval_sr(
             'srmed + a * min(lf, b) * (shf - c) - d * (max(e, cube(sef)) - f) - g * lhf',
             dict(RESID, a=2.6748257, b=0.19116789, c=0.062100567, d=0.28368908,
                  e=-0.43502578, f=0.69079, g=0.14703362))))),
]

for c in candidates:
    c['r2'] = get_r2(c['pred_mm'])
    print(f"{c['label']:16s}  R² = {c['r2']:.4f}")


In [ ]:
baselines = [c for c in candidates if c['label'] in ('SR-MED', 'SR-HI', 'NN-GAUSS')]
new_runs  = [c for c in candidates if c not in baselines]
ordered   = sorted(new_runs, key=lambda c: c['r2']) + baselines[::-1]

labels = [c['label'] for c in ordered]
r2s    = [c['r2']   for c in ordered]
colors = [c['color'] for c in ordered]

fig, ax = pplt.subplots(figwidth=4.5, figheight=0.25*len(ordered)+0.6)
ax.barh(labels, r2s, color=colors, edgecolor='none')

r2_srmed   = next(c['r2'] for c in candidates if c['label'] == 'SR-MED')
r2_srhi    = next(c['r2'] for c in candidates if c['label'] == 'SR-HI')
r2_nngauss = next(c['r2'] for c in candidates if c['label'] == 'NN-GAUSS')
ax.axvline(r2_srmed,   color='#D42028', linestyle='--', linewidth=0.8, zorder=0)
ax.axvline(r2_srhi,    color='#8B0000', linestyle='--', linewidth=0.8, zorder=0)
ax.axvline(r2_nngauss, color='#2355a1', linestyle='--', linewidth=0.8, zorder=0)

for i, r2 in enumerate(r2s):
    ax.text(r2 + 0.003, i, f'{r2:.3f}', va='center', ha='left', fontsize=8)

ax.format(grid=False, xlabel=r'$R^2$ (native mm, test split)',
          xlim=(0, r2_nngauss + 0.12),
          title='sr_gauss_resid candidates vs baselines')
pplt.show()


In [ ]:
import pandas as pd

table_rows = [
    dict(label='SR-MED',         complexity=16, run='—',
         equation='a·cube(max(rh, thetae − b·θ★ − c))'),
    dict(label='SR-HI',          complexity=23, run='—',
         equation='cube(a·max(rh, thetae + b·θ★ + c) + max(lf,shf)·d)'),
    dict(label='NN-GAUSS',       complexity='—', run='—',
         equation='Gaussian kernel NN'),

    dict(label='resid c19 (v1)',  complexity=19, run='(old resid)',
         equation='srmed + min((exp(shf)−1.10)·(lf−1.21), 1.18−lhf)'),
    dict(label='gap c20 (v1)',    complexity=20, run='(old gap)',
         equation='srmed + min(lf, 0.107)·shf·(1+srmed) − 0.185·lhf'),
    dict(label='distil c20 (v1)', complexity=20, run='(old distil)',
         equation='cube(max(rh, thetae − 1.484·θ★ − 0.281) + 0.137·rh)'),

    dict(label='s42 c20',        complexity=20, run='sr_gauss_resid s42',
         equation='srmed + 2.97·shf·min(lf,0.091) − max(−0.16, sef−0.91)'),
    dict(label='s42 c30',        complexity=30, run='sr_gauss_resid s42',
         equation='srmed + 2.73·shf·min(lf,0.169) + 0.265·(max(cube(sef), −(sdo+0.57)) − 0.69)'),
    dict(label='s72 c23',        complexity=23, run='sr_gauss_resid s72',
         equation='srmed − (0.091−shf)·cube(lf−0.624) − 0.123·cube(lf)·sef'),
    dict(label='s72 c30',        complexity=30, run='sr_gauss_resid s72',
         equation='max(0.402·srmed+0.076, srmed − (0.098−shf+0.134·cube(lf)·sef)·cube(lf−0.573))'),
    dict(label='s102 c21',       complexity=21, run='sr_gauss_resid s102',
         equation='srmed + max(−srmed, shf)·(lf−1.04) + 0.88 − max(0.70, sef)'),
    dict(label='s102 c25',       complexity=25, run='sr_gauss_resid s102',
         equation='srmed + max(−srmed, shf)·(lf−0.927) − 0.306·(max(−0.435, cube(sef)) − 0.604)'),

    dict(label='cons-lin c20',   complexity=20, run='consensus',
         equation='srmed + a·min(lf,b)·(shf−c) − d·sef'),
    dict(label='cons-cube c26',  complexity=26, run='consensus',
         equation='srmed + a·shf·min(lf,b) − c·(max(d, cube(sef)) − e)'),
    dict(label='cons-full c30',  complexity=30, run='consensus',
         equation='cons-cube + linear −g·lhf term'),
]

r2_lookup = {c['label']: c['r2'] for c in candidates}
for row in table_rows:
    row['R²'] = f"{r2_lookup[row['label']]:.4f}" if row['label'] in r2_lookup else '—'

df = (pd.DataFrame(table_rows)
        .rename(columns={'label':'Label','complexity':'Complexity','run':'Run','equation':'Equation'})
        .set_index('Label'))
display(df)
